In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="HOzmwRbpl40Q2jXmfokB")
project = rf.workspace("carla-xwaym").project("carla-hal4w-e2szl")
version = project.version(1)
dataset = version.download("coco")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to CARLA--1 in coco:: 100%|██████████| 9338/9338 [00:01<00:00, 5392.91it/s]


In [7]:
import torch
if torch.cuda.is_available():
    print(f" SUCCESS! PyTorch found the GPU: {torch.cuda.get_device_name(0)}")
else:
    print(" FAILED! PyTorch is still stuck on CPU.")

 SUCCESS! PyTorch found the GPU: Tesla T4


In [8]:
!pip install -q transformers albumentations accelerate onnx onnxscript pycocotools

import torch
import os
import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from transformers import YolosImageProcessor, YolosForObjectDetection, TrainingArguments, Trainer
import albumentations as A

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Point to the extracted Roboflow dataset automatically!
train_img_dir = f"{dataset.location}/train"
train_ann_file = f"{dataset.location}/train/_annotations.coco.json"

print("Parsing Roboflow COCO Annotations...")
coco_temp = COCO(train_ann_file)
cats = coco_temp.loadCats(coco_temp.getCatIds())
CARLA_CLASSES = [cat['name'] for cat in cats]
print(f"Detected Classes: {CARLA_CLASSES}")

id2label = {i: name for i, name in enumerate(CARLA_CLASSES)}
label2id = {name: i for i, name in enumerate(CARLA_CLASSES)}

# 2. Load Pretrained YOLOS-Tiny
model_id = "hustvl/yolos-tiny"
processor = YolosImageProcessor.from_pretrained(model_id)
model = YolosForObjectDetection.from_pretrained(
    model_id,
    num_labels=len(CARLA_CLASSES),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model.to(device)

# 3. UPGRADED AUGMENTATIONS (Mimicking "RandAugment" for Bounding Boxes)
# Following the paper's advice to prefer augmentation over structural regularization for mid-sized datasets
transform = A.Compose([
    A.Resize(384, 384),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.OneOf([
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        A.RandomBrightnessContrast(p=1.0),
    ], p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=3),
        A.GaussNoise(var_limit=(10.0, 50.0)),
    ], p=0.3),
], bbox_params=A.BboxParams(format="coco", label_fields=["category"], clip=True, min_visibility=0.2))

# 4. Robust PyTorch Dataset
class CarlaCocoDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, annotation_file, processor, transform=None):
        self.image_dir = image_dir
        self.coco = COCO(annotation_file)
        self.image_ids = list(self.coco.imgs.keys())
        self.processor = processor
        self.transform = transform
        self.coco_to_model_id = {coco_id: i for i, coco_id in enumerate(self.coco.getCatIds())}

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.image_dir, img_info['file_name'])

        img = Image.open(img_path).convert("RGB")
        img_np = np.array(img)

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        boxes, categories = [], []
        for ann in anns:
            x, y, w, h = ann['bbox']
            if w > 2 and h > 2:
                x = max(0, min(x, img_info['width'] - 1))
                y = max(0, min(y, img_info['height'] - 1))
                w = max(1, min(w, img_info['width'] - x))
                h = max(1, min(h, img_info['height'] - y))
                boxes.append([x, y, w, h])
                categories.append(self.coco_to_model_id[ann['category_id']])

        if self.transform and len(boxes) > 0:
            transformed = self.transform(image=img_np, bboxes=boxes, category=categories)
            img_np = transformed['image']
            boxes = transformed['bboxes']
            categories = transformed['category']

        if len(boxes) == 0:
            boxes = [[10.0, 10.0, 10.0, 10.0]]
            categories = [0]
            if self.transform:
                transformed = self.transform(image=img_np, bboxes=boxes, category=categories)
                img_np = transformed['image']

        formatted_annotations = []
        for box, cat in zip(boxes, categories):
            formatted_annotations.append({
                "image_id": img_id,
                "category_id": cat,
                "bbox": box,
                "area": box[2] * box[3],
                "iscrowd": 0
            })

        target = {"image_id": img_id, "annotations": formatted_annotations}
        encoding = self.processor(images=img_np, annotations=target, return_tensors="pt")

        pixel_values = encoding["pixel_values"].squeeze(0)
        labels = encoding["labels"][0]

        return {"pixel_values": pixel_values, "labels": labels}

def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = [item["labels"] for item in batch]
    return {"pixel_values": pixel_values, "labels": labels}

print("\nLoading Dataset into Memory...")
train_dataset = CarlaCocoDataset(train_img_dir, train_ann_file, processor, transform)

# 5. UPGRADED TRAINING ARGUMENTS (Based on ViT Paper Methodology)
training_args = TrainingArguments(
    output_dir="./yolos_carla_robust",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,  # Simulates a larger batch size
    num_train_epochs=3,
    fp16=False,
    save_steps=1000,
    logging_steps=50,
    learning_rate=1e-4,
    # Applied Paper Methodology Below:
    lr_scheduler_type="cosine",     # Cosine learning rate schedule
    warmup_steps=300,               # Linear warmup phase
    weight_decay=0.03,              # Decoupled weight decay of 0.03
    max_grad_norm=1.0,              # Gradient clipping at global norm 1
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset
)

print("\n--- Starting Methodological CARLA Fine-Tuning ---")
trainer.train()

# 6. Export to ONNX
print("\nExporting optimized model to ONNX...")
dummy_input = torch.randn(1, 3, 384, 384, device=device)
onnx_filename = "yolos_carla_robust_384.onnx"
torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    input_names=["pixel_values"],
    output_names=["logits", "pred_boxes"],
    opset_version=18
)

from google.colab import files
print(f"Successfully exported {onnx_filename}!")
files.download(onnx_filename)
if os.path.exists(f"{onnx_filename}.data"):
    files.download(f"{onnx_filename}.data")

Using device: cuda
Parsing Roboflow COCO Annotations...
loading annotations into memory...
Done (t=0.12s)
creating index...
index created!
Detected Classes: ['carla-car-pedestrian-trafficLight', 'ambulance', 'bike', 'car', 'firetruck', 'motobike', 'motorcycle', 'person', 'police', 'traffic_light_green', 'traffic_light_orange', 'traffic_light_red', 'traffic_sign_30', 'traffic_sign_60', 'traffic_sign_90', 'truck', 'vehicle']


preprocessor_config.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

[transformers] You passed `num_labels=17` which is incompatible to the `id2label` map of length `91`.


model.safetensors: reconstructing file:   0%|          |  0.00B / 26.0MB            

...

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Successfully exported yolos_carla_robust_384.onnx!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>